[//]: # (cr:doc name='chapter_c02_archetype_derivation' id=046ea43d)
# Chapter c02: Archetype Derivation (Causal Track)

Heart of the causal track:
1. Loads the `@production` model + the gold features.
2. Freezes a stratified SHAP background sample.
3. Computes per-row SHAP via partition-wise `pandas_udf` (TreeExplainer with the frozen background).
4. Pre-selects top features by mean |SHAP| (`KMEANS_FEATURE_CAP`).
5. Runs a silhouette-swept Spark KMeans over the reduced SHAP space.
6. Fits per-cluster surrogate trees → JSON predicates.
7. Maps archetypes ↔ playbooks via feature-overlap baseline + optional LLM refinement.
8. Writes `archetype_catalog` + `eligibility_policy` rows as `pending_review` for the c03 approval gate.


In [ ]:
# @cr:code name='init_progress' id=64ff16db
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("c02_archetype_derivation.ipynb")
# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


[//]: # (cr:doc name='c02_configuration' id=aa85033a)
## Configuration

The cell below is the only place you should need to edit. Every value here is read by the derivation cell — nothing is hardcoded inside the algorithm.

- **`LLM_ENDPOINT_NAME`** — Mosaic AI Foundation Model endpoint used to propose the archetype → playbook mapping and write rationales. Default `databricks-claude-sonnet-4-6` is pay-per-token and pre-configured in every Databricks workspace. Set to `""` to run the deterministic prose-overlap matcher only (no network calls). The setup cell below lists every foundation-model endpoint your workspace exposes so you can swap.
- **`FIT_AUTO_THRESHOLD`** — prose-match fit score (0-1) at or above which an (archetype, playbook) policy row is eligible for auto-promotion in c03. Default `0.5`.
- **`FIT_REVIEW_THRESHOLD`** — fit score at or above which a policy row is written as `pending_review` (human approves). Below this it is dropped as a regular row. Default `0.2`.
- **`DEFAULT_PLAYBOOK_ID`** — catch-all playbook id. If an archetype produces no auto/review matches, one policy row is emitted pointing at this playbook so every archetype is served. Set `""` to disable; c02 then flags uncovered archetypes in the summary and `c05` will fail on them.
- **SHAP attribution** — importance vector + background means are persisted as `shap_attribution.json` on the training run (see `stages.modeling.shap_attribution`). c02 loads that artifact via `MODEL_URI` — no sample-size knob here, no rescoring.
- **`KMEANS_K_RANGE` / `KMEANS_MAX_K`** — silhouette sweep range; the chosen `k` is capped to keep clusters human-reviewable.
- **`KMEANS_FEATURE_CAP`** — pre-select this many top features by mean absolute SHAP before clustering. Mandatory on Spark Connect: KMeans serializes the trained model and >100 columns can exceed the 1 GB serialization limit. 50 is a safe interpretable default.
- **`FORCE_DERIVATION`** — re-run derivation even when an `active` archetype already exists for the current model version. Useful after editing the playbook catalog or the surrogate-tree depth.


In [ ]:
# @cr:config name='configuration' id=120e5dff
LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-6"

FIT_AUTO_THRESHOLD = 0.5
FIT_REVIEW_THRESHOLD = 0.2
DEFAULT_PLAYBOOK_ID = ""

KMEANS_K_RANGE = (4, 12)
KMEANS_MAX_K = 8
KMEANS_FEATURE_CAP = 50

FORCE_DERIVATION = False


[//]: # (cr:doc name='c02_archetype_derivation_setup' id=66695767)
## 2.0 Setup

Resolves catalog / schema / model identifiers from `ScoringConfig` (reads the persisted Databricks init JSON on Databricks, or the local pipeline's `best_model_meta.json` for local runs). The composite-name-qualified gold features table name is derived here so the algorithmic cells stay free of path-construction logic.


In [ ]:
# @cr:code name='setup_and_resolve_model' id=37c68a6e
from customer_retention.core.compat.detection import get_spark_session, is_databricks
from customer_retention.core.config import get_playbooks_dir
from customer_retention.core.config.experiments import get_experiments_dir
from customer_retention.stages.scoring import ScoringConfig

spark = get_spark_session()
PLAYBOOKS_DIR = get_playbooks_dir()

if is_databricks():
    scoring_config = ScoringConfig.from_databricks()
    CATALOG = scoring_config.catalog
    SCHEMA = scoring_config.schema
    MODEL_NAME = scoring_config.registered_model_name
    import mlflow

    mlflow_client = mlflow.tracking.MlflowClient()
    production_version = mlflow_client.get_model_version_by_alias(MODEL_NAME, "production")
    MODEL_VERSION = production_version.version
    MODEL_URI = f"models:/{MODEL_NAME}@production"
else:
    scoring_config = ScoringConfig.from_local_config(get_experiments_dir())
    CATALOG = "local"
    SCHEMA = "local"
    MODEL_NAME = scoring_config.best_model_name or "local_model"
    MODEL_VERSION = "local"
    MODEL_URI = None

COMPOSITE_NAME = scoring_config.composite_name
GOLD_FEATURES_FQN = (
    f"{CATALOG}.{SCHEMA}.gold_features_{COMPOSITE_NAME}"
    if COMPOSITE_NAME
    else f"{CATALOG}.{SCHEMA}.gold_features"
)

ARCHETYPE_CATALOG_FQN = f"{CATALOG}.{SCHEMA}.archetype_catalog"
ELIGIBILITY_POLICY_FQN = f"{CATALOG}.{SCHEMA}.eligibility_policy"
DECISION_POLICY_FQN = f"{CATALOG}.{SCHEMA}.decision_policy"
ELIGIBILITY_SNAPSHOT_FQN = f"{CATALOG}.{SCHEMA}.eligibility_snapshot"
PREDICTIONS_FQN = f"{CATALOG}.{SCHEMA}.predictions"
TOP_SHAP_DRIVERS_FQN = f"{CATALOG}.{SCHEMA}.top_shap_drivers"

print(f"Resolved playbooks_dir: {PLAYBOOKS_DIR}")
print(f"Catalog/schema:         {CATALOG}.{SCHEMA}")
print(f"Composite name:         {COMPOSITE_NAME or '(unset)'}")
print(f"Gold features table:    {GOLD_FEATURES_FQN}")
print(f"Model URI:              {MODEL_URI or '(local)'}")
print(f"Model version:          {MODEL_VERSION}")


[//]: # (cr:doc name='c02_derive_section' id=1b5967b7)
## 2.1 Derive Archetypes + Eligibility Policies


In [ ]:
# @cr:code name='derive_archetypes_and_policies' id=23ab67ed
from customer_retention.stages.causal import (
    DerivationConfig,
    FitThresholds,
    build_llm_namer,
    derive_archetypes_and_policies,
)
from customer_retention.stages.causal.playbook_loader import load_playbooks_from_dir


def _list_foundation_model_endpoints():
    """Return (configured endpoint reachable?, list of available endpoint names)."""
    try:
        from mlflow.deployments import get_deploy_client
        client = get_deploy_client("databricks")
        endpoints = client.list_endpoints() or []
    except Exception as exc:
        print(f"(Could not list serving endpoints: {exc})")
        return False, []
    names = sorted({str(e.get("name", "")) for e in endpoints if e.get("name")})
    kind_hints = ("claude", "llama", "gpt", "mistral", "dbrx", "mixtral", "gemma")
    foundation = [n for n in names if any(h in n.lower() for h in kind_hints)]
    return True, foundation


def _model_version_already_derived(_spark, table_fqn, model_name, model_version):
    if _spark is None or not _spark.catalog.tableExists(table_fqn):
        return False
    df = _spark.sql(
        f"SELECT 1 FROM {table_fqn} "
        "WHERE model_name = ? AND model_version = ? AND status = 'active' LIMIT 1",
        args=[model_name, model_version],
    )
    return df.limit(1).count() > 0


# Transparency: show which endpoint is configured and what else is available.
print(f"Configured LLM endpoint: {LLM_ENDPOINT_NAME or '(deterministic prose_overlap only)'}")
_reachable, _available = _list_foundation_model_endpoints()
if _reachable:
    if LLM_ENDPOINT_NAME and LLM_ENDPOINT_NAME not in _available:
        print(
            f"WARNING: {LLM_ENDPOINT_NAME!r} is not in the reachable foundation model list. "
            "If matching silently falls back to prose_overlap, that is why."
        )
    print("Available foundation-model endpoints on this workspace:")
    for _name in _available:
        marker = " *" if _name == LLM_ENDPOINT_NAME else "  "
        print(f"{marker} {_name}")
    print("(set LLM_ENDPOINT_NAME in the config cell above to switch)")

derivation_result = None
if not is_databricks():
    print("SKIPPED: derivation requires a Spark cluster (Databricks-only cell)")
elif not FORCE_DERIVATION and _model_version_already_derived(
    spark, ARCHETYPE_CATALOG_FQN, MODEL_NAME, MODEL_VERSION
):
    print(f"SKIPPED: active archetypes already exist for {MODEL_NAME} v{MODEL_VERSION}")
else:
    training_df = spark.table(GOLD_FEATURES_FQN)
    feature_columns = [
        c for c in training_df.columns
        if c not in ("account_id", "entity_id", "target", "churn_probability",
                     "event_timestamp", "inference_point_in_time", "model_uri")
    ]
    join_key = "account_id" if "account_id" in training_df.columns else "entity_id"
    catalog_rows, _ = load_playbooks_from_dir(PLAYBOOKS_DIR)
    llm_namer = build_llm_namer(LLM_ENDPOINT_NAME)
    print(f"Resolved LLM matcher: {llm_namer.model_id}")

    cfg = DerivationConfig(
        spark=spark,
        training_df=training_df,
        raw_feature_df=training_df,
        feature_columns=feature_columns,
        model_uri=MODEL_URI,
        target_column="target",
        join_key=join_key,
        archetype_catalog_fqn=ARCHETYPE_CATALOG_FQN,
        eligibility_policy_fqn=ELIGIBILITY_POLICY_FQN,
        playbooks=catalog_rows,
        gold_feature_names=feature_columns,
        model_name=MODEL_NAME,
        model_version=MODEL_VERSION,
        k_range=KMEANS_K_RANGE,
        k_cap=KMEANS_MAX_K,
        feature_cap=KMEANS_FEATURE_CAP,
        llm_endpoint_name=LLM_ENDPOINT_NAME,
        llm_namer=llm_namer,
        fit_thresholds=FitThresholds(
            auto=float(FIT_AUTO_THRESHOLD),
            review=float(FIT_REVIEW_THRESHOLD),
        ),
        default_playbook_id=(DEFAULT_PLAYBOOK_ID or None),
    )
    derivation_result = derive_archetypes_and_policies(cfg)
    print(derivation_result.summary())
    # Per-archetype coverage breakdown so reviewers see exactly which
    # archetypes hit auto/review/catch_all tiers or have no coverage.
    for _arch_id, _tiers in derivation_result.coverage_report().items():
        print(f"  archetype {_arch_id}: tiers={_tiers or ['(none)']}")


[//]: # (cr:doc name='c02_backfill_prose_section' id=2e42d59c)
## 2.2 Backfill `eligibility_rules_prose`


In [ ]:
# @cr:code name='backfill_eligibility_rules_prose' id=04993f07
# Re-render `eligibility_rules_prose` for every existing active policy row whose
# prose column is NULL — derivation populates the column at row-creation time
# only, so rows written before column_descriptions / feature_meta /
# feature_population_stats sidecars existed stay NULL until we rewrite them.
# Cycle 013 P4 surfaced exactly this gap; the backfill closes it idempotently.
if is_databricks() and spark is not None and spark.catalog.tableExists(ELIGIBILITY_POLICY_FQN):
    from customer_retention.analysis.auto_explorer.run_namespace import RunNamespace
    from customer_retention.stages.causal.interpretation import (
        backfill_eligibility_prose,
    )

    # Resolve the namespace locally so this cell runs standalone — the
    # derivation cell only defines `_enrich_ns` on the freshly-derive path,
    # so a backfill-only rerun (active archetypes already exist) would
    # otherwise NameError on the reference below.
    _backfill_ns = globals().get("_enrich_ns")
    if _backfill_ns is None:
        try:
            _backfill_ns = RunNamespace.from_env_or_latest(get_experiments_dir())
        except Exception as _ns_exc:
            print(f"Backfill namespace lookup failed ({type(_ns_exc).__name__}: {_ns_exc}) — proceeding without sidecar context.")
            _backfill_ns = None

    _backfill = backfill_eligibility_prose(
        spark, ELIGIBILITY_POLICY_FQN, namespace=_backfill_ns,
    )
    print(_backfill.summary())
    if _backfill.warnings:
        print("(interpretation-layer warnings — see logs for details)")
        for _w in _backfill.warnings:
            print(f"  - {_w}")


In [ ]:
# @cr:code name='release_stage_memory' id=b26ca247
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
